# The `Retriever` class

Everything from Day 31-32 (chunking -> TF-IDF -> Chroma) wrapped into one reusable class.

Three methods:

1. `chunker(document)` - takes a document (string) and cuts it into overlapping chunks
2. `embedder(chunks)` - turns the chunks into vectors and loads them into a Chroma collection
3. `search(query, k)` - embeds the question and does a cosine similarity search

In [6]:
import chromadb
from sklearn.feature_extraction.text import TfidfVectorizer

## The class

The three methods run in order and each one hands its output to the next:

`document -> chunks -> vectors in Chroma -> search results`

The chunks and the fitted vectorizer are stored on `self`, so `search()` can reuse the exact
same vocabulary that the chunks were embedded with. That part matters - a TF-IDF vector only
means something if the question and the chunks were translated by the *same* vectorizer.

In [7]:
class Retriever:
    """Chunk a document, embed the chunks, and search them with cosine similarity."""

    def __init__(self, chunk_size=1000, overlap=100, collection_name="retriever"):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.collection_name = collection_name

        self.chunks = []
        self.vectorizer = TfidfVectorizer()
        self.client = chromadb.Client()   # in-memory database
        self.collection = None

    # ==========================================
    # 1. CHUNKER
    # ==========================================
    def chunker(self, document):
        """Cut the document into overlapping windows of `chunk_size` characters.

        The overlap keeps a sentence from being sliced in half and losing its meaning.
        """
        if self.overlap >= self.chunk_size:
            raise ValueError("overlap must be smaller than chunk_size, else start never moves")

        chunks = []
        start = 0
        while start < len(document):
            end = start + self.chunk_size
            chunk = document[start:end].strip()
            if chunk:                      # skip windows that are only whitespace
                chunks.append(chunk)
            start += self.chunk_size - self.overlap

        self.chunks = chunks
        return chunks

    # ==========================================
    # 2. EMBEDDER
    # ==========================================
    def embedder(self, chunks=None, metadatas=None):
        """Fit TF-IDF on the chunks and push the vectors into a Chroma collection."""
        if chunks is not None:
            self.chunks = chunks
        if not self.chunks:
            raise ValueError("no chunks yet - call chunker() first")

        # Learn the vocabulary from the chunks, then turn them into dense rows
        embeddings = self.vectorizer.fit_transform(self.chunks).toarray()

        # Start the collection fresh so re-running this cell doesn't stack duplicates
        try:
            self.client.delete_collection(self.collection_name)
        except Exception:
            pass

        self.collection = self.client.create_collection(
            name=self.collection_name,
            metadata={"hnsw:space": "cosine"},   # cosine similarity, not euclidean
        )

        self.collection.add(
            ids=[str(i) for i in range(len(self.chunks))],
            embeddings=embeddings.tolist(),
            documents=self.chunks,
            metadatas=metadatas,
        )

        return embeddings

    # ==========================================
    # 3. SEARCH
    # ==========================================
    def search(self, query, k=3, where=None):
        """Embed the query with the *same* vectorizer and pull the k closest chunks."""
        if self.collection is None:
            raise ValueError("nothing indexed yet - call embedder() first")

        query_vector = self.vectorizer.transform([query]).toarray()[0]

        # TF-IDF only knows words it saw in the chunks. A question made entirely of
        # unseen words becomes an all-zero vector, and cosine distance on that is undefined.
        if not query_vector.any():
            print(f"none of the words in {query!r} appear in the document")
            return []

        results = self.collection.query(
            query_embeddings=[query_vector.tolist()],
            n_results=min(k, len(self.chunks)),
            where=where,
        )

        # Chroma hands back a dict of lists (one list per query) - unpack the first one
        hits = []
        for doc, dist, chunk_id in zip(
            results["documents"][0], results["distances"][0], results["ids"][0]
        ):
            hits.append({
                "id": chunk_id,
                "text": doc,
                "distance": dist,
                "score": 1 - dist,      # cosine distance -> similarity
            })
        return hits

## Trying it out

Small document so the chunk boundaries are easy to eyeball. `chunk_size` is tiny here on purpose -
for a real PDF go back to something like 1000 with 100 overlap.

In [8]:
document = """
Retrieval Augmented Generation gives a language model access to documents it was never trained on.
The document is split into chunks because a whole book will not fit inside the context window.
Each chunk is converted into a vector, which is just a list of numbers describing the text.
TF-IDF builds those numbers from word counts, weighted down for words that appear everywhere.
The vectors are stored in a vector database such as Chroma, which indexes them for fast lookup.
At query time the question is embedded with the same vectorizer and compared against every chunk.
Cosine similarity measures the angle between two vectors, so length of the text does not dominate.
The closest chunks are pasted into the prompt, and the model answers using that retrieved context.
"""

retriever = Retriever(chunk_size=220, overlap=40)

chunks = retriever.chunker(document)
print(f"{len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"------ Chunk {i} ------")
    print(chunk, "\n")

5 chunks

------ Chunk 0 ------
Retrieval Augmented Generation gives a language model access to documents it was never trained on.
The document is split into chunks because a whole book will not fit inside the context window.
Each chunk is converted i 

------ Chunk 1 ------
ontext window.
Each chunk is converted into a vector, which is just a list of numbers describing the text.
TF-IDF builds those numbers from word counts, weighted down for words that appear everywhere.
The vectors are sto 

------ Chunk 2 ------
t appear everywhere.
The vectors are stored in a vector database such as Chroma, which indexes them for fast lookup.
At query time the question is embedded with the same vectorizer and compared against every chunk.
Cosin 

------ Chunk 3 ------
and compared against every chunk.
Cosine similarity measures the angle between two vectors, so length of the text does not dominate.
The closest chunks are pasted into the prompt, and the model answers using that retrie 

------ Chunk

In [9]:
embeddings = retriever.embedder()

print("embedding matrix:", embeddings.shape)   # (n_chunks, vocabulary_size)
print("vocabulary size :", len(retriever.vectorizer.vocabulary_))
print("in collection   :", retriever.collection.count())

embedding matrix: (5, 98)
vocabulary size : 98
in collection   : 5


In [10]:
questions = [
    "how are the vectors compared to each other?",
    "why is the document split up?",
    "what does TF-IDF do with common words?",
]

for question in questions:
    print(f"\n=== {question} ===")
    for hit in retriever.search(question, k=2):
        print(f"[{hit['score']:.3f}] {hit['text'][:120]}...")


=== how are the vectors compared to each other? ===
[0.265] and compared against every chunk.
Cosine similarity measures the angle between two vectors, so length of the text does n...
[0.226] t appear everywhere.
The vectors are stored in a vector database such as Chroma, which indexes them for fast lookup.
At ...

=== why is the document split up? ===
[0.382] Retrieval Augmented Generation gives a language model access to documents it was never trained on.
The document is split...
[0.150] ontext window.
Each chunk is converted into a vector, which is just a list of numbers describing the text.
TF-IDF builds...

=== what does TF-IDF do with common words? ===
[0.244] ontext window.
Each chunk is converted into a vector, which is just a list of numbers describing the text.
TF-IDF builds...
[0.084] t appear everywhere.
The vectors are stored in a vector database such as Chroma, which indexes them for fast lookup.
At ...


## Same class, on a real PDF

Nothing about the class changes - only the string you feed `chunker()`. Metadata is optional;
pass one dict per chunk and you can later filter with `search(..., where={"source": "syllabus"})`.

In [12]:
import os

pdf_path = r"D:\Ai engineering course\Day_25\Micro_Syallabus.pdf"

if os.path.exists(pdf_path):
    import fitz

    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text() + "\n"

    pdf_retriever = Retriever(chunk_size=1000, overlap=100, collection_name="syllabus")
    pdf_chunks = pdf_retriever.chunker(text)
    pdf_retriever.embedder(metadatas=[{"source": "syllabus"} for _ in pdf_chunks])

    for hit in pdf_retriever.search("what is covered in the course?", k=3):
        print(f"[{hit['score']:.3f}] {hit['text'][:200]}...\n")
else:
    print(f"pdf not found at {pdf_path} - skipping")

[0.288] t /
Applied AI Lead
7+ yrs
Strategy, org-wide impact
FAQ
What does AI Engineering & Machine Learning cost and how do I pay?
NPR 37,800. Pay by cash at our Old Baneshwor campus, bank transfer, or Fonep...

[0.260] s what gets Nepal-based engineers onto remote international teams
Saarathi Gate Assessment before Day 1, diagnostic, no pass or fail
Nothing to install beforehand. VS Code, Python and the OpenAI, Anth...

[0.248] SAARATHI ACADEMY
for Digital Excellence
AI Engineering & Machine Learning
COURSE BRIEF
SKILLS & TOOLS YOU WILL MASTER
Python + scikit-learn
PyTorch + Transformers
LangChain + LangGraph
PC RAG + Pineco...

